<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/cournot_5d_nonpotential_stochastic_mlp_dtb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5D constrained non-potential Cournot: stochastic neural-DTB with score evolution

## Five-player game setup

For player $i$, let $r_i=\sum_{j\ne i}x_j$. With $b=1$ and $\mu=2$,
nonnegative production gives

$$
\operatorname{BR}_i(x)=\max\{2r_i(1-r_i),0\},
\qquad
b_i(x)=2\bigl[\operatorname{BR}_i(x)-x_i\bigr].
$$

The stochastic implementation uses five independent noise components of
amplitude $0.1$:

$$
\sigma_1=\cdots=\sigma_5=0.1,
\qquad D=0.01I_5.
$$

The initial distribution is uniform on $[0,1]^5$. The thesis reports seven
reference equilibria: the origin, $(7/32)^5$, and the five permutations of
$(0,5/18,5/18,5/18,5/18)$. It reports all seven as unstable and notes that
the list may be incomplete.


## Block 0 — Shared repository setup

This notebook reuses `ResidualMLPMap`, `game_dtb_basis_matrices`, and
`map_at` from `run_game_dtb.py`; `count_trainable` from `network.py`;
and `flat_params` plus `jform_solve` from `dtb.py`. Spatial tangent
derivatives and score evolution come from the shared `utility.py`.

In [ ]:
# BLOCK 0 — Locate the shared modules locally or clone the requested branch in Colab.
from pathlib import Path
import math
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

MODULE_FILES = ('run_game_dtb.py', 'network.py', 'dtb.py', 'utility.py')
EXPERIMENT_DIR = next(
    (
        folder
        for folder in (Path.cwd(), Path.cwd() / 'DTB_Game_Ver2')
        if all((folder / name).is_file() for name in MODULE_FILES)
    ),
    None,
)

if EXPERIMENT_DIR is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run from the repository root or DTB_Game_Ver2 directory.')
    repo = Path('/content/dtb-colab-experiments')
    if not (repo / '.git').exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo),
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(repo), 'checkout', 'codex/game-dynamics-dtb'], check=True)
        subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
    EXPERIMENT_DIR = repo / 'DTB_Game_Ver2'

# Import the existing DTB, network, and map primitives requested for reuse.
sys.path.insert(0, str(EXPERIMENT_DIR.resolve()))
from run_game_dtb import ResidualMLPMap, game_dtb_basis_matrices
from network import count_trainable
from dtb import device, flat_params, jform_solve
from importlib import reload
import utility as stochastic_utility

stochastic_utility = reload(stochastic_utility)
from utility import (
    create_run_directory,
    diagnostics_markdown,
    euler_score_update,
    plot_dtb_em_snapshots,
    plot_tangent_diagnostics,
    sample_initial_with_score,
    save_run_data,
    sliced_wasserstein_distance,
    tangent_velocity_spatial_terms,
)

print('Shared module directory:', EXPERIMENT_DIR)

## Block 1 — Game and experiment controls

All numerical, neural-network, stochastic, saving, and visualization
parameters are editable below. The complete realized configuration is
printed after the particles and tangent basis have been initialized.

In [ ]:
# BLOCK 1 — Define the five-player game and every editable experiment control.
SAVE_RUN = False  # True creates a uniquely named folder under saved_runs/.
SEED = 2026
EM_SEED = SEED + 10_000

DIM = 5
N_PARTICLES = 2000
H = 0.005
T_FINAL = 0.16
N_STEPS = round(T_FINAL / H)
SNAPSHOT_TIMES = (0.0, 0.12, 0.14, 0.16)

# Independent stochastic forcing is applied to all five player quantities.
NOISE_STD = (0.1, 0.1, 0.1, 0.1, 0.1)

# "uniform" matches the paper's samples and uses the exact interior score q_0=0.
# "gaussian" and "smoothed_uniform" provide globally smooth analytical scores.
INITIAL_LAW = 'uniform'
GAUSSIAN_MEAN = 0.5
GAUSSIAN_STD = 0.15
UNIFORM_SMOOTHING_STD = 0.02

NN_CHOICE = 'mlp'
NN_ACTIVATION = 'tanh'  # Smooth activation is required by grad(div(u)).
MLP_WIDTH = 32
MLP_DEPTH = 4
BASIS_SIZE = 128
SVD_RTOL = 1e-3
SVD_METHOD = 'svd_gpu'
JACOBIAN_CHUNK = 64
DERIVATIVE_CHUNK = 16
PRINT_EVERY = 4

DTYPE = torch.float32
DEVICE = device()
COURNOT_B = 1.0
COURNOT_MU = 2.0
COORDINATE_PAIRS = ((1, 2), (3, 4))

def game_velocity(x):
    """Five-player constrained non-potential Cournot best-response field."""
    if x.shape[-1] != DIM:
        raise ValueError(f'Expected states with last dimension {DIM}.')
    rivals = x.sum(dim=-1, keepdim=True) - x
    best_response = (COURNOT_MU * rivals * (1.0 - rivals)).clamp_min(0.0)
    return 2.0 * COURNOT_B * (best_response - x)

known_points = [torch.zeros(DIM, device=DEVICE, dtype=DTYPE)]
known_points.append(torch.full((DIM,), 7/32, device=DEVICE, dtype=DTYPE))
for zero_player in range(DIM):
    point = torch.full((DIM,), 5/18, device=DEVICE, dtype=DTYPE)
    point[zero_player] = 0.0
    known_points.append(point)
KNOWN_EQUILIBRIA = torch.stack(known_points)
STABLE_MASK = np.zeros(len(KNOWN_EQUILIBRIA), dtype=bool)

## Block 2 — Particle, score, and neural tangent initialization

The persistent labels start at $x_i^0=X_0(z_i)=z_i$. Each label carries
its score $q_i^0=\nabla\log\rho_0(x_i^0)$. A single seeded subset of MLP
parameter directions defines the tangent basis for the complete run.

In [ ]:
# BLOCK 2 — Initialize particles, their scores, and the fixed neural tangent basis.
if NN_ACTIVATION not in {'tanh', 'gelu', 'silu'}:
    raise ValueError('Score evolution requires a twice-differentiable activation.')
if not np.isclose(N_STEPS * H, T_FINAL):
    raise ValueError('T_FINAL must be an integer multiple of H.')
if len(NOISE_STD) != DIM:
    raise ValueError('NOISE_STD needs one independent amplitude per player.')

# Seed the neural network and use separate recorded streams for particles and EM noise.
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
generator_device = DEVICE.type if DEVICE.type == 'cuda' else 'cpu'
initial_generator = torch.Generator(device=generator_device).manual_seed(SEED)
em_generator = torch.Generator(device=generator_device).manual_seed(EM_SEED)

# Attach the analytical initial score to every persistent particle label.
x_0, q_0, log_density_0 = sample_initial_with_score(
    N_PARTICLES,
    DIM,
    law=INITIAL_LAW,
    device=DEVICE,
    dtype=DTYPE,
    generator=initial_generator,
    gaussian_mean=GAUSSIAN_MEAN,
    gaussian_std=GAUSSIAN_STD,
    smoothing_std=UNIFORM_SMOOTHING_STD,
)
x_k = x_0.clone()
q_k = q_0.clone()
log_density_k = log_density_0.clone()
em_k = x_0.clone()  # The baseline starts from exactly the same point cloud.

# The fixed MLP provides tangent directions; it is not trained during the run.
mlp = ResidualMLPMap(
    dim=DIM,
    width=MLP_WIDTH,
    depth=MLP_DEPTH,
    activation=NN_ACTIVATION,
    dtype=DTYPE,
    zero_init_output=False,
).net.to(DEVICE)
N_PARAMETERS = count_trainable(mlp)
theta_0, structure, _ = flat_params(mlp)
mlp.requires_grad_(False)

# Select one reproducible random sub-basis and keep it fixed for all time steps.
if not isinstance(BASIS_SIZE, int) or not 1 <= BASIS_SIZE <= N_PARAMETERS:
    raise ValueError(f'BASIS_SIZE must be in [1, {N_PARAMETERS}].')
basis_generator = torch.Generator(device='cpu').manual_seed(SEED)
selected = torch.randperm(N_PARAMETERS, generator=basis_generator)[:BASIS_SIZE]
selected = selected.sort().values.to(DEVICE)

# Sigma is the SDE amplitude; D=Sigma Sigma^T is the Fokker--Planck covariance.
sigma = torch.tensor(NOISE_STD, device=DEVICE, dtype=DTYPE)
diffusion = torch.diag(sigma.square())

# Verify that every plotted analytical reference is stationary for this game field.
equilibrium_residual = float(game_velocity(KNOWN_EQUILIBRIA).norm(dim=1).max())
if equilibrium_residual > 2e-5:
    raise ValueError(f'An equilibrium reference has drift residual {equilibrium_residual:.3e}.')

# Convert requested physical times to exact stored state indices.
snapshot_steps = [round(value / H) for value in SNAPSHOT_TIMES]
if any(not np.isclose(step * H, value) for step, value in zip(snapshot_steps, SNAPSHOT_TIMES)):
    raise ValueError('Every snapshot time must lie on the time grid.')

# Save every experiment setting and the exact selected parameter indices.
CONFIG = {
    'game_dimension': DIM,
    'particles': N_PARTICLES,
    'step_size': H,
    'steps': N_STEPS,
    'final_time': T_FINAL,
    'snapshot_times': list(SNAPSHOT_TIMES),
    'noise_std': list(NOISE_STD),
    'diffusion_diagonal': sigma.square().detach().cpu().tolist(),
    'maximum_equilibrium_residual': equilibrium_residual,
    'initial_law': INITIAL_LAW,
    'gaussian_mean': GAUSSIAN_MEAN,
    'gaussian_std': GAUSSIAN_STD,
    'uniform_smoothing_std': UNIFORM_SMOOTHING_STD,
    'network': NN_CHOICE,
    'activation': NN_ACTIVATION,
    'width': MLP_WIDTH,
    'depth': MLP_DEPTH,
    'trainable_parameters': N_PARAMETERS,
    'basis_size': BASIS_SIZE,
    'selected_parameter_indices': selected.detach().cpu().tolist(),
    'svd_rtol': SVD_RTOL,
    'svd_method': SVD_METHOD,
    'jacobian_chunk': JACOBIAN_CHUNK,
    'derivative_chunk': DERIVATIVE_CHUNK,
    'coordinate_pairs': [list(pair) for pair in COORDINATE_PAIRS],
    'seed': SEED,
    'em_seed': EM_SEED,
    'device': str(DEVICE),
    'dtype': str(DTYPE),
    'save_run': SAVE_RUN,
    'output_root': str(EXPERIMENT_DIR / 'saved_runs'),
}

# Print a complete, readable statement immediately after initialization.
print('=' * 76)
print(f'STOCHASTIC {DIM}D COURNOT NEURAL-DTB EXPERIMENT')
print('=' * 76)
print(f'Game/player dimension: d={DIM}; b={COURNOT_B}; mu={COURNOT_MU}')
print(f'Particles and time: N={N_PARTICLES}; h={H}; K={N_STEPS}; T={T_FINAL}')
print(f'Snapshot times: {SNAPSHOT_TIMES}')
print(f'Noise amplitudes sigma_i: {NOISE_STD}')
print(f'Diffusion covariance diagonal D_ii: {CONFIG["diffusion_diagonal"]}')
print(f'Maximum equilibrium drift residual: {equilibrium_residual:.3e}')
print(f'Initial law: {INITIAL_LAW}; Gaussian mean/std={GAUSSIAN_MEAN}/{GAUSSIAN_STD}; smoothing={UNIFORM_SMOOTHING_STD}')
print(f'Initial score RMS: {float(q_0.square().sum(dim=1).mean().sqrt()):.6g}')
print(f'Network: {NN_CHOICE}; {DIM} -> ' + ' -> '.join([str(MLP_WIDTH)] * MLP_DEPTH) + f' -> {DIM}')
print(f'Activation/depth/width: {NN_ACTIVATION}/{MLP_DEPTH}/{MLP_WIDTH}')
print(f'Trainable parameters: {N_PARAMETERS}')
print(f'Selected tangent size: {BASIS_SIZE}; selection seed={SEED}')
print(f'Selected flat-parameter indices: {CONFIG["selected_parameter_indices"]}')
print(f'Projection solver: {SVD_METHOD}; relative SVD tolerance={SVD_RTOL}')
print(f'Jacobian/derivative chunks: {JACOBIAN_CHUNK}/{DERIVATIVE_CHUNK}')
print(f'Device/dtype: {DEVICE}/{DTYPE}; EM seed={EM_SEED}')
print(f'Coordinate planes: {COORDINATE_PAIRS}')
print(f'Save run: {SAVE_RUN}; output root={CONFIG["output_root"]}')
print('Reported reference equilibria:')
for index, point in enumerate(KNOWN_EQUILIBRIA.detach().cpu().numpy()):
    stability = 'stable' if STABLE_MASK[index] else 'unstable'
    print(f'  E_{index} = {np.array2string(point, precision=8)} [{stability}]')
if INITIAL_LAW == 'uniform':
    print('Uniform-score note: q_0=0 is the exact interior score; the boundary score is singular.')
print('=' * 76)

## Block 3 — Stochastic DTB tangent and score update

At step $k$, the Fokker--Planck transport target is

$$
g_{k,i}=b(x_{k,i})-\frac12Dq_{k,i}.
$$

The selected neural tangent basis is projected onto this target:

$$
\alpha_k=\arg\min_\alpha\sum_i\|J_{k,i}^S\alpha-g_{k,i}\|_2^2,
\qquad u_k(x)=\partial_{\theta_S}f_{\theta_0}(x)\alpha_k.
$$

Particles and attached scores use the same old-step velocity:

$$
x_i^{k+1}=x_i^k+h u_k(x_i^k),
$$

$$
q_i^{k+1}=q_i^k-h\left([D_xu_k(x_i^k)]^\mathsf{T}q_i^k
+\nabla_x[\nabla_x\!\cdot u_k](x_i^k)\right).
$$

This is the ordinary fixed-basis DTB update: there is no resampling, refit,
score network, or post-run polish step.

In [ ]:
# BLOCK 3 — Evolve the DTB particles/scores and the matched Euler--Maruyama cloud.
state_times = np.arange(N_STEPS + 1, dtype=float) * H
solve_times = np.arange(N_STEPS, dtype=float) * H

# Histories are copied to CPU after each step to keep GPU memory bounded.
dtb_history = [x_k.detach().cpu().numpy()]
score_history = [q_k.detach().cpu().numpy()]
log_density_history = [log_density_k.detach().cpu().numpy()]
em_history = [em_k.detach().cpu().numpy()]
coefficient_history = []
diagnostic_records = []

run_start = time.perf_counter()
for step in range(N_STEPS):
    step_start = time.perf_counter()

    # 1. Evaluate the game drift and score-dependent diffusion correction.
    drift_k = game_velocity(x_k)
    diffusion_correction = 0.5 * (q_k @ diffusion.T)
    target_k = drift_k - diffusion_correction

    # 2. Build the selected neural tangent basis at the current physical particles.
    _, jacobian_tensor, stacked_jacobian = game_dtb_basis_matrices(
        theta_0,
        selected,
        x_k,
        mlp,
        structure,
        chunk=JACOBIAN_CHUNK,
    )

    # 3. Project the stochastic target velocity using the shared DTB SVD solver.
    alpha_k = jform_solve(
        stacked_jacobian,
        target_k.reshape(-1),
        rtol=SVD_RTOL,
        method=SVD_METHOD,
    )

    # 4. Differentiate only the final projected direction u_k=J_k alpha_k.
    tangent_k, grad_u_k, divergence_k, grad_divergence_k = tangent_velocity_spatial_terms(
        theta_0,
        selected,
        alpha_k,
        x_k,
        mlp,
        structure,
        chunk_size=DERIVATIVE_CHUNK,
    )

    # 5. Measure the tangent projection before changing particles or scores.
    residual_k = tangent_k - target_k
    target_norm = target_k.norm().clamp_min(1e-30)
    relative_error = residual_k.norm() / target_norm
    particle_scale = math.sqrt(N_PARTICLES)

    # 6. Update the score with (D_x u_k)^T q_k and grad(div(u_k)).
    q_next, transported_score, score_source = euler_score_update(
        q_k,
        grad_u_k,
        grad_divergence_k,
        H,
    )

    # 7. Apply the ordinary DTB particle-map Euler step using the same old u_k.
    x_next = x_k + H * tangent_k
    log_density_next = log_density_k - H * divergence_k

    # 8. Advance the matched direct SDE baseline from the same old-time state.
    em_noise = torch.randn(
        em_k.shape,
        device=DEVICE,
        dtype=DTYPE,
        generator=em_generator,
    )
    em_next = em_k + H * game_velocity(em_k) + math.sqrt(H) * sigma * em_noise

    # 9. Record the requested tangent metrics and score/diffusion health values.
    diagnostic_records.append({
        'time': step * H,
        'relative_projection_error': float(relative_error),
        'alpha_norm': float(alpha_k.norm()),
        'score_rms': float(q_k.norm() / particle_scale),
        'diffusion_rms': float(diffusion_correction.norm() / particle_scale),
        'target_rms': float(target_k.norm() / particle_scale),
        'tangent_rms': float(tangent_k.norm() / particle_scale),
        'score_transport_rms': float(transported_score.norm() / particle_scale),
        'score_source_rms': float(score_source.norm() / particle_scale),
        'seconds': time.perf_counter() - step_start,
    })
    coefficient_history.append(alpha_k.detach().cpu().numpy())

    # 10. Detach the new state because no gradient graph is propagated across time.
    x_k = x_next.detach()
    q_k = q_next.detach()
    log_density_k = log_density_next.detach()
    em_k = em_next.detach()

    # 11. Store full histories so any requested time can be visualized afterward.
    dtb_history.append(x_k.cpu().numpy())
    score_history.append(q_k.cpu().numpy())
    log_density_history.append(log_density_k.cpu().numpy())
    em_history.append(em_k.cpu().numpy())

    # 12. Fail immediately if the explicit score evolution becomes non-finite.
    if not all(torch.isfinite(value).all() for value in (x_k, q_k, log_density_k, em_k)):
        raise FloatingPointError(f'Non-finite state detected after step {step + 1}.')
    if (step + 1) % PRINT_EVERY == 0 or step in (0, N_STEPS - 1):
        record = diagnostic_records[-1]
        print(
            f'k={step + 1:4d}/{N_STEPS}  t={(step + 1) * H:.4f}  '
            f'rel.err={record["relative_projection_error"]:.3e}  '
            f'||alpha||={record["alpha_norm"]:.3e}  '
            f'score.rms={record["score_rms"]:.3e}'
        )

dtb_history = np.stack(dtb_history)
score_history = np.stack(score_history)
log_density_history = np.stack(log_density_history)
em_history = np.stack(em_history)
coefficient_history = np.stack(coefficient_history)
CONFIG['wall_seconds'] = time.perf_counter() - run_start
print(f'Completed {N_STEPS} stochastic DTB steps in {CONFIG["wall_seconds"]:.2f} seconds.')

## Block 4 — Matched snapshots and diagnostics

Snapshot columns use identical physical times and axis limits. Neural-DTB
appears in the first row and the matched Euler--Maruyama baseline in the
second. The tangent figure shows the relative projection error and
$\|\alpha_k\|_2$ across time. No PCA is used.

In [ ]:
# BLOCK 4 — Plot matched snapshots, plot tangent diagnostics, and optionally save the run.
output_dir = None
if SAVE_RUN:
    output_dir = create_run_directory(
        EXPERIMENT_DIR / 'saved_runs',
        dim=DIM,
        network=NN_CHOICE,
        activation=NN_ACTIVATION,
        width=MLP_WIDTH,
        depth=MLP_DEPTH,
        parameter_count=N_PARAMETERS,
        basis_size=BASIS_SIZE,
        seed=SEED,
    )
    CONFIG['output_dir'] = str(output_dir)

# Each coordinate plane gets a two-row figure: DTB above and EM below.
snapshot_figures = plot_dtb_em_snapshots(
    dtb_history,
    em_history,
    state_times,
    snapshot_steps,
    coordinate_pairs=COORDINATE_PAIRS,
    equilibria=KNOWN_EQUILIBRIA,
    stable_mask=STABLE_MASK,
    output_dir=output_dir,
)

# Plot exactly the two requested tangent-bundle diagnostics against physical time.
relative_projection_errors = np.asarray([
    record['relative_projection_error'] for record in diagnostic_records
])
alpha_norms = np.asarray([record['alpha_norm'] for record in diagnostic_records])
diagnostics_figure = plot_tangent_diagnostics(
    solve_times,
    relative_projection_errors,
    alpha_norms,
    output_path=None if output_dir is None else output_dir / 'tangent_diagnostics.png',
)
plt.show()

# Display the diagnostic definitions and their first, mean, and final values in math.
metric_table = diagnostics_markdown(diagnostic_records)
display(Markdown('## Stochastic DTB diagnostics'))
display(Markdown(metric_table))

# Compare final point-cloud location and spread without reducing dimension by PCA.
final_mean_gap = np.linalg.norm(dtb_history[-1].mean(axis=0) - em_history[-1].mean(axis=0))
final_covariance_gap = np.linalg.norm(
    np.cov(dtb_history[-1], rowvar=False) - np.cov(em_history[-1], rowvar=False)
)
final_sliced_wasserstein = sliced_wasserstein_distance(
    dtb_history[-1], em_history[-1], seed=SEED
)
comparison_table = '\n'.join([
    '| Final DTB versus Euler--Maruyama metric | Value |',
    '| :--- | ---: |',
    rf'| $\|\bar x_{{\mathrm{{DTB}}}}-\bar x_{{\mathrm{{EM}}}}\|_2$ | {final_mean_gap:.4e} |',
    rf'| $\|\operatorname{{Cov}}_{{\mathrm{{DTB}}}}-\operatorname{{Cov}}_{{\mathrm{{EM}}}}\|_F$ | {final_covariance_gap:.4e} |',
    f'| Sliced $W_2$ | {final_sliced_wasserstein:.4e} |',
])
display(Markdown('## Final distribution comparison'))
display(Markdown(comparison_table))

# SAVE_RUN=False leaves the repository unchanged and only displays results.
if output_dir is not None:
    save_run_data(
        output_dir,
        config=CONFIG,
        arrays={
            'times': state_times,
            'solve_times': solve_times,
            'dtb_particles': dtb_history,
            'scores': score_history,
            'log_density': log_density_history,
            'em_particles': em_history,
            'coefficients': coefficient_history,
            'selected_parameter_indices': selected.detach().cpu().numpy(),
            'relative_projection_error': relative_projection_errors,
            'alpha_norm': alpha_norms,
        },
        metrics_markdown=metric_table + '\n\n' + comparison_table,
    )
    print('Saved figures, histories, configuration, and metric tables to:', output_dir)
else:
    print('SAVE_RUN=False: results were displayed without creating an output folder.')